# Imports

In [1]:
# Local application/library specific imports
from pygrex.config import cfg
from pygrex.data_reader import DataReader, GroupInteractionHandler
# from pygrex.evaluator import SlidingWindowEvaluator
from pygrex.explain import SlidingWindowExplainer
from pygrex.models import ALS
from pygrex.recommender import GroupRecommender
from pygrex.utils import SlidingWindow, AggregationStrategy
from pygrex.evaluator import run_evaluation_with_proper_split


import time
import pandas as pd


In [2]:
# Read the ratings file.
data = DataReader(**cfg.data.test)
data.make_consecutive_ids_in_dataset()
data.binarize(binary_threshold=1)

# Read the file with the group ids
group_handler = GroupInteractionHandler(**cfg.data.groups)
available_groups = group_handler.read_groups("groupsWithHighRatings5.txt")
print("✅ Data preparation complete.\n")

# --- Display Data Summary ---
print("--- Data Summary ---")
print(f"👥 Unique Users: {data.num_user:,}")
print(f"📦 Unique Items: {data.num_item:,}")
print(f"⭐ Total Ratings: {len(data.get_raw_dataset()):,}")
print(f"👨‍👩‍👧‍👦 Number of Groups: {len(available_groups):,}")
print("\nProcessed Ratings DataFrame Head:")
display(data.dataset.head())

✅ Data preparation complete.

--- Data Summary ---
👥 Unique Users: 610
📦 Unique Items: 9,724
⭐ Total Ratings: 100,836
👨‍👩‍👧‍👦 Number of Groups: 17

Processed Ratings DataFrame Head:


,userId,itemId,rating,timestamp
0,0,0,1,964982703
1,0,2,1,964981247
2,0,5,1,964982224
3,0,43,1,964983815
4,0,46,1,964982931


## Step 2: Model Training & Evaluation

With the data prepared, we now select and train a recommendation model. We will use **Alternating Least Squares (ALS)**, a matrix factorization technique for implicit feedback. After training, we will evaluate its performance using a train/test split to measure its Hit Ratio and NDCG.

In [3]:
print("--- 2.1 Model Training ---")

# Train the recommendation model
model = ALS(**cfg.model.als)

# Train the model
start_time = time.time()
model.fit(data)
end_time = time.time()
training_time = end_time - start_time

print(f"✅ Model trained successfully in {training_time:.2f} seconds!")

--- 2.1 Model Training ---


c:\Users\usuar\miniconda3\envs\pygrex-exp-grs\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/10 [00:00<?, ?it/s]

✅ Model trained successfully in 1.08 seconds!


In [ ]:
print("\n--- 2.2 Offline Model Evaluation ---")
# For evaluation, a new model instance must be created.
# The evaluation function handles its own internal data splitting and training.
eval_model = ALS(**cfg.model.als)

# Define evaluation parameters
test_size = 0.2
top_n = 10

print(f"Running evaluation with a {test_size*100:.0f}% test split (Top-{top_n})...")

# Run the evaluation
evaluation_scores = run_evaluation_with_proper_split(
    data_reader=data,
    model=eval_model,
    test_size=test_size,
    top_n=top_n,
)

# Display evaluation results
print("\n--- Evaluation Results ---")
print(f"Hit Ratio @{top_n}: {evaluation_scores.get('Hit Ratio', 0.0):.2%}")
print(f"NDCG @{top_n}: {evaluation_scores.get('NDCG', 0.0):.4f}")
print(f"Evaluation Time: {evaluation_scores.get('evaluation_time', 0):.1f}s")

## Step 3: Group Recommendation

Now that we have a trained model, we can generate recommendations for a group. We will select a group, choose an aggregation strategy to combine individual member preferences, and generate a Top-10 list of recommended items.

In [7]:
print("--- 3. Group Recommendation ---")

# Select a group and strategy
selected_group_id = available_groups[0]  # Let's use the first group as an example
group_members = group_handler.parse_group_members(selected_group_id)
aggregation_strategy = AggregationStrategy.AVG_PREDICTIONS # Use the simple average strategy
top_k = 10

print(f"Generating Top-{top_k} recommendations for group: {selected_group_id}")
print(f"👥 Group Members: {group_members}")
print(f"📊 Aggregation Strategy: {aggregation_strategy.name}")

# --- Generate Recommendations ---
# 1. Instantiate the GroupRecommender
group_recommender = GroupRecommender(data=data)

# 2. Setup the recommendation process
group_recommender.setup_recommendation(
    model=model,
    members=group_members,  # type: ignore
    data=data,
    aggregation_strategy=aggregation_strategy,
                )


# 3. Get the final recommendation list
recommended_items = group_recommender.get_group_recommendations(top_k=top_k)
recommendation_scores = group_recommender.get_recommendation_scores()

print("\n✅ Recommendations generated successfully!")

# --- Display Results ---
rec_data = [
    {
        "Rank": i + 1,
        "Item ID": item_id,
        "Aggregated Score": recommendation_scores.get(item_id, 0.0),
    }
    for i, item_id in enumerate(recommended_items)  # type: ignore
]

rec_df = pd.DataFrame(rec_data)
print(f"\nTop {top_k} Recommended Items:")
display(rec_df)

--- 3. Group Recommendation ---
Generating Top-10 recommendations for group: 522_385_234_452_594
👥 Group Members: [522, 385, 234, 452, 594]
📊 Aggregation Strategy: AVG_PREDICTIONS

✅ Recommendations generated successfully!

Top 10 Recommended Items:


,Rank,Item ID,Aggregated Score
0,1,543,4.636274
1,2,757,4.582981
2,3,564,4.504107
3,4,441,4.488708
4,5,379,4.341830
5,6,475,4.279482
6,7,43,4.268454
7,8,19,4.225248
8,9,748,4.178329
9,10,64,4.147735


## Step 4: Explanation (Sliding Window)

Finally, we generate an explanation for one of the recommendations. We will use the **Sliding Window** method to find a counterfactual explanation. This method answers the question: *"Which minimal set of items, if removed from the group's history, would cause our target item to disappear from the recommendation list?"*


In [8]:
print("--- 4. Counterfactual Explanation (Sliding Window) ---")

# Select a target item from our recommendation list to explain
target_item = recommended_items[0]
# Configure the explainer
window_size = 3
# These weights determine how to rank items from the group's history
# before attempting to remove them to find an explanation.
ranking_weights = {
    "popularity": 1.0,
    "intensity": 1.0,
    "rating": 1.0,
    "relevance": 1.0,
    "trend": 1.0,
}

print(f"Generating explanation for recommended item: {target_item}")
print(f"Sliding Window Size: {window_size}\n")

# --- Generate Explanation ---
# 1. Get all items previously rated by the group
items_rated_by_group = group_handler.get_rated_items_by_all_group_members(
    group=group_members, original_data=data
)

# 2. Instantiate the explainer
explainer = SlidingWindowExplainer(
    config=cfg, # Not needed for this explainer
    data=data,
    group_handler=group_handler,
    members=group_members,
    target_item=target_item,
    aggregation_strategy=aggregation_strategy,
    model=model,
    window_size=window_size,
)

# 3. Find the explanation
explanations = explainer.find_explanation(
    items_rated_by_group=items_rated_by_group,
    group_predictions=group_recommender.get_individual_predictions(),
    top_recommendation=group_recommender.get_top_recommendation(),
    ranking_weights=ranking_weights,
)

--- 4. Counterfactual Explanation (Sliding Window) ---
Generating explanation for recommended item: 543
Sliding Window Size: 3



  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

If the group had not interacted with these items [np.int64(480)],
the item of interest 543 would not have appeared on the recommendation list;
instead, 303 would have been recommended.
